# BSRNN 구조가 컴파일 비용을 어떻게 바꾸는가 — LSTM 이 비싼 이유 찾기

**묻는 것** — `torch.compile` 이 BSRNN 에서 오래 걸리는 **원인이 무엇인가.**
모델 구조(`num_repeat` · `feature_dim`)를 바꿔 가며 컴파일 시간이 **무엇에 비례하는지** 봄.

| # | 물음 | 절 |
|---|---|---|
| 1 | LSTM 이 몇 개이고 무엇을 받는가 | 1 |
| 2 | 컴파일 시간이 **깊이(`num_repeat`)에 비례**하나 | 2 |
| 3 | 컴파일 시간이 **폭(`feature_dim`)에 비례**하나 | 3 |
| 4 | 그 시간은 **separator 몫인가 ECAPA 몫인가** | 4 |
| 5 | 그래프는 **어디서 끊기나** | 5 |
| 6 | **LSTM 을 실제로 컴파일**하면(`allow_rnn=True`) 어떻게 되나 | 6 |

yaml 설정 그대로 두고 **돌리는 방식**만 흔드는 쪽은
[bsrnn_compile_axes_yaml.ipynb](bsrnn_compile_axes_yaml.ipynb) 가 맡음.

## 0. 준비 — 측정 코드는 `_bench/bench_common.py` 에 있음

이 노트북은 **측정을 직접 하지 않음.** 워커가 GPU 여러 장에서 재어
[_runs/](_runs/) 에 쌓아 둔 결과를 **읽어서 표만 그림.**

| 무엇 | 어디 |
|---|---|
| 측정 코드 (모델 · 로더 모사 · `bench()` · 조건 목록) | [_bench/bench_common.py](_bench/bench_common.py) |
| 워커 | [_bench/worker.py](_bench/worker.py) |
| 결과 | `_runs/<tag>/<조건해시>.json` — 조건 하나에 파일 하나 |

워커를 띄우는 법 (GPU 3장 기준):

```bash
cd wesep/notebooks/_bench
CUDA_VISIBLE_DEVICES=0 python worker.py --worker 0 --n-workers 3
CUDA_VISIBLE_DEVICES=1 python worker.py --worker 1 --n-workers 3
CUDA_VISIBLE_DEVICES=3 python worker.py --worker 2 --n-workers 3
```

`scope`·`dynamic` 만 다른 조건은 **한 묶음으로 같은 GPU** 에 감 —
`speedup` 의 기준이 되는 `eager` 와 떨어지면 비교가 깨지기 때문임.

## 측정 규약

| 항목 | 값 |
|---|---|
| 웜업 | **10스텝** — 등록 길이가 매 스텝 달라지므로 재컴파일을 여기서 끝냄 |
| 측정 | **50스텝**. **`ms_step_median`**(중앙값)과 **`ms_step_mean`**(평균)을 함께 남김 |
| 컴파일 시간 | **첫 스텝**의 벽시계 시간. 조건마다 `force_disable_caches = True` 로 **콜드** 보장 |
| 데이터 | **wesep 데이터로더 모사** — `wav_mix` 는 고정, 등록 발화는 배치 최소 길이로 잘림 |

| 표 읽는 법 | |
|---|---|
| 왼쪽 | **통제변인** — `scope` · `dynamic` · `b` · `t` · `prec` · `tf32_lstm` · `num_repeat` · `feature_dim` · `allow_rnn` · `gpu` |
| 오른쪽 | **측정값** — `compile_s` · `ms_step_median` · `ms_step_mean` · `peak_GiB` · `graph_break` · `dynamo_frames` · `recompile_in_measure` · `status` |

`dynamo_frames` 는 dynamo 가 컴파일한 **프레임 수**, `graph_break` 는 그래프가 **끊긴 횟수**로 서로 다른 값임.
**`recompile_in_measure` 가 0 이 아니면 웜업이 모자라 측정 구간에 재컴파일이 섞인 것**이므로 그 행은 믿지 말 것.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "_bench"))
import bench_common as bc

display(bc.env_table())
display(bc.conf_table())

/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/kaldiio/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/s3prl/upstream/byol_s/byol_a/common.py:20: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("sox_io")
ESPnet is not installed, cannot use espnet_hubert upstream


/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,항목,값,비고
0,torch,2.7.1+cu128,cuda 12.8
1,DEVICE,cuda,모든 측정이 이 장치에서 돎
2,GPU,NVIDIA GeForce RTX 3090,23.6 GiB
3,CUDA_VISIBLE_DEVICES,0,이 GPU 에 다른 프로세스가 있으면 측정이 오염됨
4,wesep 루트,/workspace/git_clone/SD-FiLM/wesep,sys.path 에 넣음
5,결과 CSV,/workspace/git_clone/SD-FiLM/wesep/notebooks/_...,조건 하나가 한 줄


,항목,값,뜻
0,dataloader_args.batch_size,8,혼합 개수. 모델이 보는 배치는 이것의 2배
1,모델 배치 (실효),16,tse_collate_fn 이 편 뒤의 b
2,dataset_args.chunk_len,48000,입력 길이 t (샘플) = 3.0 초
3,model_args.num_repeat,6,BSNet 반복 수. LSTM 개수는 이것의 2배
4,model_args.feature_dim,128,밴드당 채널 c
5,fbank num_mel_bins,80,등록 발화 fbank 의 f
6,clip_grad,5.0,executor.py 가 매 스텝 부름
7,ECAPA 가중치 존재,True,/workspace/git_clone/SD-FiLM/wesep/examples/li...


### 0.1 wesep 데이터로더 모사 — 왜 등록 발화만 길이가 변하나

[tse_collate_fn](../wesep/dataset/dataset.py#L206-L264) 이 하는 일 중 shape 에 관계된 것만 옮겼음.

| # | 로더가 하는 일 | 결과 |
|---|---|---|
| 1 | 샘플마다 화자 2명분 `wav_mix` 를 복제 | 모델 배치 `b` = yaml `batch_size` × 2 |
| 2 | `wav_mix` 는 `chunk_len` 으로 이미 잘려 있음 | **separator 입력은 항상 고정** |
| 3 | 등록 발화는 화자마다 길이가 달라 `mode="min"` 이 **배치 최소 길이로 잘라냄** | **`spk_model` 입력은 배치마다 달라짐** |

그래서 `separator` 는 정적 shape, `spk_model`(ECAPA)은 동적 shape 을 받음 — 컴파일 조건이 서로 반대임.

**개별 발화 길이 분포는 근사이고 미검증임.** 실측한 것은 `b=16` 일 때 `min` 의 분포(299~1005 · 중앙값 396)뿐이라,
개별 길이를 `[299, 1950]` 균등으로 잡아 그 `min` 이 실측과 맞도록 역산했음.

In [2]:
display(bc.enroll_probe_table())

,b,등록 T 최소,등록 T 중앙값,등록 T 최대,고유 개수(60스텝)
0,1,346,1105,1930,60
1,2,315,793,1794,58
2,4,302,555,1700,60
3,8,300,430,907,60
4,16,300,362,761,54


### 0.2 측정 진행 상황

워커가 도는 중에도 이 셀만 다시 돌리면 어디까지 갔는지 보임.
`남음` 이 0 이 아닌 태그의 표는 **아직 일부만 그려진 것**임.

In [3]:
display(bc.progress_table())

,tag,조건 수,측정됨,남음,진행
0,structure-depth,15,15,0,100 %
1,structure-width,6,6,0,100 %
2,structure-who,5,5,0,100 %
3,structure-allow-rnn,4,4,0,100 %
4,axes-base,7,7,0,100 %
5,axes-single,10,10,0,100 %
6,axes-full,210,210,0,100 %


---

## 1. LSTM 이 몇 개이고 무엇을 받는가

이 절만 **모델을 직접 만들어** 확인함(측정이 아니라 구조 확인이라 GPU 1장이면 몇 초).
[BSNet.forward](../wesep/models/bsrnn.py#L69-L85) 가 텐서를 두 번 접어 넣기 때문에
같은 `ResRNN` 이라도 **시퀀스 축이 서로 다름** — `band_rnn` 은 프레임, `band_comm` 은 밴드임.

In [4]:
import torch
import pandas as pd

probe = bc.build_bsrnn()

lstm_rows = [{"모듈 경로": n, "input_size": m.input_size, "hidden_size": m.hidden_size,
              "num_layers": m.num_layers, "bidirectional": m.bidirectional,
              "params_M": round(sum(p.numel() for p in m.parameters()) / 1e6, 3)}
             for n, m in probe.named_modules() if isinstance(m, torch.nn.LSTM)]
df_lstm = pd.DataFrame(lstm_rows)
display(df_lstm)

total_M = sum(p.numel() for p in probe.parameters()) / 1e6
display(pd.DataFrame([
    {"항목": "LSTM 개수", "값": len(df_lstm), "뜻": f"num_repeat {bc.YAML_NREP} x ResRNN 2"},
    {"항목": "num_layers 고유값", "값": str(sorted(df_lstm["num_layers"].unique())),
     "뜻": "1 뿐이면 nn.LSTM 자체의 적층은 없음"},
    {"항목": "LSTM 파라미터 (M)", "값": round(df_lstm["params_M"].sum(), 2),
     "뜻": "모델 전체 {:.2f} M 의 {:.0f} %".format(
         total_M, df_lstm["params_M"].sum() / total_M * 100)},
]))

,모듈 경로,input_size,hidden_size,num_layers,bidirectional,params_M
0,separator.separation.1.band_rnn.rnn,128,256,1,True,0.791
1,separator.separation.1.band_comm.rnn,128,256,1,True,0.791
2,separator.separation.2.band_rnn.rnn,128,256,1,True,0.791
3,separator.separation.2.band_comm.rnn,128,256,1,True,0.791
4,separator.separation.3.band_rnn.rnn,128,256,1,True,0.791
5,separator.separation.3.band_comm.rnn,128,256,1,True,0.791
6,separator.separation.4.band_rnn.rnn,128,256,1,True,0.791
7,separator.separation.4.band_comm.rnn,128,256,1,True,0.791
8,separator.separation.5.band_rnn.rnn,128,256,1,True,0.791
9,separator.separation.5.band_comm.rnn,128,256,1,True,0.791


,항목,값,뜻
0,LSTM 개수,12,num_repeat 6 x ResRNN 2
1,num_layers 고유값,[1],1 뿐이면 nn.LSTM 자체의 적층은 없음
2,LSTM 파라미터 (M),9.49,모델 전체 27.64 M 의 34 %


In [5]:
shape_rows = []
handles = [m.register_forward_hook(
    lambda mod, inp, out, n=n: shape_rows.append(
        {"모듈 경로": n, "b": inp[0].shape[0], "seq": inp[0].shape[1], "c": inp[0].shape[2],
         "dtype": str(inp[0].dtype).replace("torch.", "")}))
    for n, m in probe.named_modules() if isinstance(m, torch.nn.LSTM)]

with torch.no_grad(), torch.amp.autocast(bc.DEVICE.type, enabled=True, dtype=torch.float16):
    probe(*bc.make_wesep_batch(bc.YAML_B, bc.YAML_T, 0)[:2])

for h in handles:
    h.remove()

df_shape = pd.DataFrame(shape_rows)
display(df_shape)
display(df_shape.groupby(["seq", "c", "b"]).size().reset_index(name="LSTM 개수"))

del probe
import gc
gc.collect()
bc.empty_cache()

,모듈 경로,b,seq,c,dtype
0,separator.separation.1.band_rnn.rnn,512,376,128,float32
1,separator.separation.1.band_comm.rnn,6016,32,128,float32
2,separator.separation.2.band_rnn.rnn,512,376,128,float32
3,separator.separation.2.band_comm.rnn,6016,32,128,float32
4,separator.separation.3.band_rnn.rnn,512,376,128,float32
5,separator.separation.3.band_comm.rnn,6016,32,128,float32
6,separator.separation.4.band_rnn.rnn,512,376,128,float32
7,separator.separation.4.band_comm.rnn,6016,32,128,float32
8,separator.separation.5.band_rnn.rnn,512,376,128,float32
9,separator.separation.5.band_comm.rnn,6016,32,128,float32


,seq,c,b,LSTM 개수
0,32,128,6016,6
1,376,128,512,6


---

## 2. 깊이 축 — `num_repeat`

`num_repeat` 만 바꿈. **LSTM 개수는 그 2배**임.
[ResRNN:23-29](../wesep/models/bsrnn.py#L23-L29) 의 `nn.LSTM(..., 1, ...)` 이 **1로 박혀 있어**
`num_layers` 는 config 로 못 바꿈 — 깊이를 바꾸는 손잡이는 `num_repeat` 하나뿐임.

컴파일 시간이 깊이에 **선형이면** 그래프 크기가 비용을 정하는 것이고,
**평평하면** 비용이 깊이와 무관한 다른 곳(예: ECAPA)에 있다는 뜻임.

In [6]:
rows_depth = bc.run_or_load("structure-depth")
display(bc.as_table(rows_depth, keep=("num_repeat", "lstm_n")))

ok = pd.DataFrame(rows_depth)
ok = ok[(ok["status"] == "ok") & (ok["scope"] != "eager")] if len(ok) else ok
if len(ok):
    ok = ok.assign(compile_s_per_repeat=(ok["compile_s"] / ok["num_repeat"]).round(2),
                   ms_per_lstm=(ok["ms_step_median"] / ok["lstm_n"]).round(1))
    display(ok[["scope", "num_repeat", "lstm_n", "compile_s", "compile_s_per_repeat",
                "ms_step_median", "ms_per_lstm", "peak_GiB"]])

,scope,dynamic,num_repeat,lstm_n,gpu,compile_s,ms_step_median,ms_step_mean,peak_GiB,graph_break,dynamo_frames,recompile_in_measure,status,error
0,eager,-,1,2,0,0.0,263.4,263.1,4.98,0,0,0,ok,NaN
1,separator,True,1,2,0,24.9,249.8,251.7,4.97,3,9,0,ok,NaN
2,separator&spk,True,1,2,0,36.8,251.0,254.0,4.97,3,10,0,ok,NaN
3,eager,-,2,4,1,0.0,326.1,329.6,7.86,0,0,0,ok,NaN
4,separator,True,2,4,1,25.1,299.6,302.9,7.85,4,10,0,ok,NaN
5,separator&spk,True,2,4,1,37.5,297.5,300.9,7.85,4,11,0,ok,NaN
6,eager,-,4,8,3,0.0,475.4,475.4,13.62,0,0,0,ok,NaN
7,separator,True,4,8,3,25.9,454.4,459.4,13.62,4,10,0,ok,NaN
8,separator&spk,True,4,8,3,37.5,452.4,453.2,13.62,4,11,0,ok,NaN
9,eager,-,6,12,0,0.0,615.0,620.1,19.38,0,0,0,ok,NaN


,scope,num_repeat,lstm_n,compile_s,compile_s_per_repeat,ms_step_median,ms_per_lstm,peak_GiB
1,separator,1,2,24.9,24.90,249.8,124.9,4.97
2,separator&spk,1,2,36.8,36.80,251.0,125.5,4.97
4,separator,2,4,25.1,12.55,299.6,74.9,7.85
5,separator&spk,2,4,37.5,18.75,297.5,74.4,7.85
7,separator,4,8,25.9,6.48,454.4,56.8,13.62
8,separator&spk,4,8,37.5,9.38,452.4,56.6,13.62
10,separator,6,12,14.1,2.35,627.1,52.3,19.38
11,separator&spk,6,12,48.3,8.05,632.5,52.7,19.38


---

## 3. 폭 축 — `feature_dim`

밴드당 채널 수. LSTM 의 `input_size` 가 되고 `hidden_size` 는 그 2배임(1절 표).
**파라미터 수는 `feature_dim` 의 제곱에 가깝게** 늘지만 **그래프의 노드 수는 안 변함** —
컴파일 시간이 파라미터를 따라가는지 노드 수를 따라가는지 여기서 갈림.

In [7]:
rows_width = bc.run_or_load("structure-width")
display(bc.as_table(rows_width, keep=("feature_dim",)))

par = []
for fd in bc.WIDTHS:
    m = bc.build_bsrnn(feature_dim=fd)
    par.append({"feature_dim": fd,
                "params_M": round(sum(p.numel() for p in m.parameters()) / 1e6, 2)})
    del m
    bc.empty_cache()
display(pd.DataFrame(par))

,scope,dynamic,feature_dim,gpu,compile_s,ms_step_median,ms_step_mean,peak_GiB,graph_break,dynamo_frames,recompile_in_measure,status,error
0,eager,-,64,0,0.0,325.9,330.3,10.39,0,0,0,ok,NaN
1,separator,True,64,0,15.2,329.0,332.8,10.39,4,10,0,ok,NaN
2,eager,-,128,1,0.0,596.0,599.7,19.46,0,0,0,ok,NaN
3,separator,True,128,1,27.5,610.0,613.5,19.33,4,10,0,ok,NaN
4,eager,-,256,3,NaN,NaN,NaN,NaN,0,0,0,OOM,NaN
5,separator,True,256,3,NaN,NaN,NaN,NaN,0,0,0,OOM,NaN


,feature_dim,params_M
0,64,11.74
1,128,27.64
2,256,90.51


---

## 4. 그 시간은 누구 몫인가 — separator 대 ECAPA

`separator` 만 걸었을 때와 `spk_model` 까지 걸었을 때의 **차이가 곧 ECAPA 몫**임.
`spk_model` 만 거는 조건(`spk_only`)도 넣어 교차 확인함.

In [8]:
rows_who = bc.run_or_load("structure-who")
display(bc.as_table(rows_who))

w = pd.DataFrame(rows_who)
if len(w):
    w = w.set_index("scope")
    need = {"separator", "separator&spk", "spk_only"}
    if need <= set(w.index):
        display(pd.DataFrame([
            {"항목": "separator 컴파일", "초": w.loc["separator", "compile_s"]},
            {"항목": "spk_model 단독 컴파일", "초": w.loc["spk_only", "compile_s"]},
            {"항목": "둘 다", "초": w.loc["separator&spk", "compile_s"]},
            {"항목": "차이(둘 다 − separator) = ECAPA 몫",
             "초": round(w.loc["separator&spk", "compile_s"] - w.loc["separator", "compile_s"], 1)},
            {"항목": "model (한꺼번에)", "초": w.loc["model", "compile_s"]
             if "model" in w.index else float("nan")},
        ]))

,scope,dynamic,compile_s,ms_step_median,ms_step_mean,peak_GiB,graph_break,dynamo_frames,recompile_in_measure,status,error
0,eager,-,0.0,619.6,624.1,19.38,0,0,0,ok,NaN
1,separator,True,13.7,622.0,623.6,19.38,4,10,0,ok,NaN
2,spk_only,True,24.8,622.6,627.0,19.38,0,4,0,ok,NaN
3,separator&spk,True,50.4,636.5,641.8,19.38,4,11,0,ok,NaN
4,model,True,218.1,582.2,584.5,19.34,5,12,0,ok,NaN


,항목,초
0,separator 컴파일,13.7
1,spk_model 단독 컴파일,24.8
2,둘 다,50.4
3,차이(둘 다 − separator) = ECAPA 몫,36.7
4,model (한꺼번에),218.1


---

## 5. 그래프는 어디서 끊기나

`torch._dynamo.explain()` 이 **끊긴 지점과 이유를 원문 그대로** 돌려줌.
이 절만 `print` 로 원문을 남김 — 표로 뭉개면 어느 줄에서 끊겼는지가 사라지기 때문임.
이 절도 모델을 직접 만들어 돌리므로 **GPU 가 필요함.**

In [9]:
import torch._dynamo as dynamo

dynamo.reset()
dynamo.utils.counters.clear()

m = bc.build_bsrnn()
# 그래프가 끊기는 위치는 배치 크기와 무관하므로 b 를 낮춰 잼 —
# explain 은 no_grad 가 없어 b=16 이면 활성값만으로 24 GiB 를 넘김(실측 OOM).
EXPLAIN_B = 2
x, e, y = bc.make_wesep_batch(EXPLAIN_B, bc.YAML_T, 0)
explanation = dynamo.explain(m)(x, e)

print("graph 개수      :", explanation.graph_count)
print("graph break 개수:", explanation.graph_break_count)
print("op 개수         :", explanation.op_count)
print()
for i, reason in enumerate(explanation.break_reasons):
    print(f"--- break {i} ---")
    print(str(reason)[:600])
    print()

del m
bc.empty_cache()

graph 개수      : 6
graph break 개수: 5
op 개수         : 1087

--- break 0 ---
GraphCompileReason(reason='TorchDynamo purposely graph breaks on RNN, GRU, LSTMs', user_stack=[<FrameSummary file /workspace/git_clone/SD-FiLM/wesep/wesep/models/bsrnn.py, line 362 in forward>, <FrameSummary file /workspace/git_clone/SD-FiLM/wesep/wesep/models/bsrnn.py, line 146 in forward>, <FrameSummary file /workspace/git_clone/SD-FiLM/wesep/wesep/models/bsrnn.py, line 73 in forward>, <FrameSummary file /workspace/git_clone/SD-FiLM/wesep/wesep/models/bsrnn.py, line 41 in forward>], graph_break=True)

--- break 1 ---
GraphCompileReason(reason='Data dependent operator\n  Explanation: Operator `aten._local_scalar_dense.default` has a non-Tensor output whose value is dependent on the data of Tensor inputs.\n  Hint: Enable tracing of data-dependent output operators with `torch._dynamo.config.capture_scalar_outputs = True`\n\n  Developer debug context: aten._local_scalar_dense.default\n', user_stack=[<FrameSummary f

In [10]:
# 끊긴 이유를 종류별로 세어 표로도 남김
dynamo.reset()
dynamo.utils.counters.clear()

m = bc.build_bsrnn()
m.compile(dynamic=True)
for i in range(3):
    with torch.no_grad(), torch.amp.autocast(bc.DEVICE.type, enabled=True, dtype=torch.float16):
        m(*bc.make_wesep_batch(EXPLAIN_B, bc.YAML_T, i)[:2])

gb = dynamo.utils.counters.get("graph_break", {})
display(pd.DataFrame([{"이유": str(k)[:110], "횟수": v} for k, v in gb.items()]
                     or [{"이유": "(없음)", "횟수": 0}]).sort_values("횟수", ascending=False))

del m
bc.empty_cache()

/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/torch/_inductor/lowering.py:1917: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(


,이유,횟수
1,"TorchDynamo purposely graph breaks on RNN, GRU...",3
0,Data dependent operator\n Explanation: Operat...,2


---

## 6. LSTM 을 실제로 컴파일하면 — `torch._dynamo.config.allow_rnn`

기본값은 `False` 라 dynamo 가 **`nn.LSTM` 을 만나면 일부러 그래프를 끊음.**
`True` 로 켜면 LSTM 도 traced 되지만 **시점마다 펼쳐져 컴파일이 급격히 길어짐** —
그것이 BSRNN 컴파일 비용의 상한을 정하는 요인인지 여기서 확인함.

**컴파일이 수 분 걸릴 수 있어** `num_repeat` 을 1·2 로 낮춰 잼.

In [11]:
rows_rnn = bc.run_or_load("structure-allow-rnn")
display(bc.as_table(rows_rnn, keep=("allow_rnn", "num_repeat", "lstm_n")))

,scope,num_repeat,lstm_n,allow_rnn,gpu,compile_s,ms_step_median,ms_step_mean,peak_GiB,graph_break,dynamo_frames,recompile_in_measure,status,error
0,separator,1,2,False,0,14.2,249.0,250.5,4.97,3,9,0,ok,NaN
1,separator,2,4,False,1,15.2,300.5,304.7,7.85,4,10,0,ok,NaN
2,separator,1,2,True,3,NaN,NaN,NaN,NaN,0,0,0,ERROR,BackendCompilerFailed: backend='inductor' rais...
3,separator,2,4,True,0,NaN,NaN,NaN,NaN,0,0,0,ERROR,BackendCompilerFailed: backend='inductor' rais...


---

## 7. 정리 — **실행 후 채울 것**

| # | 물음 | 답 |
|---|---|---|
| 1 | 컴파일 시간이 깊이에 비례하나 | ⬜ 미측정 — 2절 `compile_s_per_repeat` 이 일정한지 |
| 2 | 폭(`feature_dim`)에 비례하나 | ⬜ 미측정 — 3절 |
| 3 | 시간의 주범은 separator 인가 ECAPA 인가 | ⬜ 미측정 — 4절 차이 |
| 4 | 그래프는 어디서 몇 번 끊기나 | ⬜ 미측정 — 5절 |
| 5 | `allow_rnn=True` 면 얼마나 늘어나나 | ⬜ 미측정 — 6절 |
| 6 | 웜업 10스텝이 충분했나 | ⬜ `recompile_in_measure` 가 전부 0 인지 |

### 이 노트북이 재지 않는 것

| 항목 | 이유 |
|---|---|
| 정확도·수렴 | 속도와 컴파일 비용만 잼 |
| `nn.LSTM(num_layers > 1)` | [ResRNN:23-29](../wesep/models/bsrnn.py#L23-L29) 에 1로 박혀 있어 **저장소를 고쳐야 함.** 이 노트북은 wesep 소스를 고치지 않음 |
| 돌리는 방식(배치·precision·tf32) | [bsrnn_compile_axes_yaml.ipynb](bsrnn_compile_axes_yaml.ipynb) 가 맡음 |